# 04A_v2 — Dynamic Pre-Challenge Fire-Approach Features

**Build: `04A_DYNAMIC_APPROACH_FEATURES_V1`**

03N froze the Held/Burned Over targets. 04A begins feature construction.

**No external downloads. No NIROPS rerasterization. No target relabeling.**

For each frozen local fireline unit, 04A reconstructs information available at or before a pre-challenge observation time:

\[
X(t\le t_0)\rightarrow Y(t>t_0)
\]

The notebook reuses the corrected unit geometries, 03L sample points and signed-distance trajectories, point evidence, and the already-downloaded NIROPS progression polygons.

## Why there is an anchor audit

03N's dominant challenge interval is ideal for target timing, but a Burned Over unit can contain several spatial samples. A smaller portion might have crossed before the dominant-weight interval. Using that later time for prediction could reveal the outcome.

Therefore 04A keeps the frozen target unchanged but compares:

- **dominant challenge start** from 03N;
- **earliest supported-event start** across the unit.

The earliest supported-event start is the candidate operational prediction anchor. A unit is flagged if any sampled portion is already inside the observed perimeter at that anchor.

## Features

- signed-distance state at \(t_0\);
- previous observed distance state;
- **observed inter-perimeter approach rate**;
- short pre-\(t_0\) trajectory trend and consistency;
- several estimated-time-to-line candidates using only \(t\le t_0\);
- local approach direction from the observed perimeter toward the fireline;
- fireline principal orientation;
- angle between the incoming approach and the fireline.

The actual \(t_1-t_0\) duration is **not** used as an ML feature.

## V2 implementation fixes

- IRWIN GUID normalization removes braces **and hyphens**.
- The known extracted NIROPS shapefile is used directly.
- 03L unit geometry is decoded from **WKB bytes stored in ordinary Parquet**.
- V2 writes to fresh output folders and does not reuse failed V1 shards.
- A one-incident preflight is included before the full 382-incident production run.


In [ ]:
import os  # Configurable project root for the public release.
NOTEBOOK_BUILD_ID = "04A_V2_DYNAMIC_APPROACH_FEATURES"

from pathlib import Path
import sys, subprocess, importlib.util

PROJECT_ROOT = Path(os.environ.get("FIRELINE_PROJECT_ROOT", "."))
CORRECTED_ROOT = PROJECT_ROOT / "data" / "processed" / "fireline_corrected_local_250m_10m"
FROZEN_ROOT = CORRECTED_ROOT / "frozen_cohort"

FROZEN_PATH = FROZEN_ROOT / "fireline_ml_cohort_frozen_v1.parquet"
DYNAMIC_PATH = FROZEN_ROOT / "fireline_ml_dynamic_timing_cohort_frozen_v1.parquet"
UNIT_ROOT = CORRECTED_ROOT / "units"
SAMPLE_ROOT = CORRECTED_ROOT / "sample_points"
POINT_EVIDENCE_ROOT = CORRECTED_ROOT / "point_evidence"
TRAJECTORY_ROOT = CORRECTED_ROOT / "trajectories"

# Exact NIROPS progression layer already extracted locally.
NIROPS_SOURCE = Path(
    (os.environ.get("FIRELINE_PROJECT_ROOT", ".") + '/data/interim/joinability_audit/nirops_extracted/NIROPS_2020_2024_R1_R6/NIROPS_2020_2024_R1_R6.shp')
)

# Fresh V2 outputs: failed V1 shards/logs are not reused.
PROCESSED_ROOT = PROJECT_ROOT / "data" / "processed" / "fireline_dynamic_approach_v2"
SHARD_ROOT = PROCESSED_ROOT / "incident_shards"
OUTPUT_ROOT = PROJECT_ROOT / "outputs" / "dynamic_approach_features_v2"
for p in [PROCESSED_ROOT, SHARD_ROOT, OUTPUT_ROOT]:
    p.mkdir(parents=True, exist_ok=True)

for p in [FROZEN_PATH, DYNAMIC_PATH, UNIT_ROOT, SAMPLE_ROOT, POINT_EVIDENCE_ROOT, TRAJECTORY_ROOT, NIROPS_SOURCE]:
    assert p.exists(), p

CRS_METRIC = "EPSG:5070"
CHALLENGE_TOLERANCE_M = 50.0
MAX_HISTORY_OBSERVATIONS = 4
DIRECTION_MIN_DISTANCE_M = 25.0
ETA_CAP_18H = 18.0
ETA_CAP_48H = 48.0
RESUME = True

print("="*88)
print("RUNNING:", NOTEBOOK_BUILD_ID)
print("NIROPS:", NIROPS_SOURCE)
print("V2 processed root:", PROCESSED_ROOT)
print("="*88)


In [ ]:
required = {
    "pandas":"pandas", "numpy":"numpy", "geopandas":"geopandas",
    "pyarrow":"pyarrow", "shapely":"shapely", "tqdm":"tqdm", "pyogrio":"pyogrio"
}
for module, package in required.items():
    if importlib.util.find_spec(module) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import json, gc, math, time, traceback
from datetime import datetime, timezone
import numpy as np
import pandas as pd
import geopandas as gpd
import shapely
from shapely.geometry import LineString, MultiLineString
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 400)
pd.set_option("display.width", 240)

print("Python:", sys.executable)
print("Pandas:", pd.__version__)
print("GeoPandas:", gpd.__version__)
print("Shapely:", shapely.__version__)

## 1. Load frozen cohorts

In [ ]:
frozen = pd.read_parquet(FROZEN_PATH)
dynamic = pd.read_parquet(DYNAMIC_PATH)

assert frozen["unit_uid"].is_unique
assert dynamic["unit_uid"].is_unique

print("Frozen units:", f"{len(frozen):,}")
print("03N dynamic-timing eligible:", f"{len(dynamic):,}")
print("Incidents:", dynamic["_irwin_norm"].nunique())

display(
    dynamic[["_engagement_norm","hold_label"]]
    .value_counts().rename("units").reset_index()
)

## 2. Correct IRWIN normalization and fixed NIROPS source

In [ ]:
def norm_irwin(s):
    """Normalize IRWIN GUIDs to lowercase 32-character hex without braces/hyphens."""
    x=(s.astype("string").str.strip()
       .str.replace("{","",regex=False)
       .str.replace("}","",regex=False)
       .str.replace("-","",regex=False)
       .str.lower())
    return x.replace({"<na>":pd.NA,"nan":pd.NA,"none":pd.NA,"":pd.NA})

# Sanity check: NIROPS GUID and processed-fireline GUID must normalize identically.
_test = pd.Series([
    "{1E0FCF1D-5F6E-454D-AC7E-901A510B3753}",
    "1E0FCF1D5F6E454DAC7E901A510B3753",
])
assert norm_irwin(_test).nunique() == 1
print("IRWIN normalization test: PASS")

NIROPS_SOURCE_RESOLVED = NIROPS_SOURCE
assert NIROPS_SOURCE_RESOLVED.exists(), NIROPS_SOURCE_RESOLVED
print("Using NIROPS:", NIROPS_SOURCE_RESOLVED)


## 3. Load NIROPS, keep the frozen incidents, and union duplicate observation times

In [ ]:
def load_nirops(path, required_irwins):
    p=Path(path)
    g = gpd.read_parquet(p) if p.suffix.lower()==".parquet" else gpd.read_file(p)
    cmap={str(c).lower():c for c in g.columns}

    irwin_col = cmap.get("_irwin_norm", cmap.get("irwinid"))
    time_col = cmap.get("_time", cmap.get("utc", cmap.get("dateutc")))
    if irwin_col is None or time_col is None:
        raise RuntimeError("Selected NIROPS source lacks expected IRWIN/time fields.")

    g["_irwin_04a"]=norm_irwin(g[irwin_col])
    g["_time_04a"]=pd.to_datetime(g[time_col], utc=True, errors="coerce")

    required_irwins=set(required_irwins)
    source_rows=len(g)
    g=g[
        g["_irwin_04a"].isin(required_irwins) &
        g["_time_04a"].notna() &
        g.geometry.notna() & ~g.geometry.is_empty
    ].copy()

    print("NIROPS source rows:", f"{source_rows:,}")
    print("Rows retained for frozen dynamic incidents:", f"{len(g):,}")
    print("Retained incidents before timestamp union:", g["_irwin_04a"].nunique())

    if g.crs is None:
        raise RuntimeError("NIROPS source has no CRS.")
    if str(g.crs).upper()!=CRS_METRIC:
        g=g.to_crs(CRS_METRIC)

    out=[]
    for irwin,grp in tqdm(g.groupby("_irwin_04a",sort=False), desc="Union NIROPS duplicate times"):
        d=grp[["_time_04a","geometry"]].dissolve(by="_time_04a").reset_index().sort_values("_time_04a")
        d["_irwin_04a"]=irwin
        out.append(d)

    return gpd.GeoDataFrame(
        pd.concat(out,ignore_index=True),
        geometry="geometry", crs=CRS_METRIC
    )[["_irwin_04a","_time_04a","geometry"]]

# dynamic was loaded in the previous cell.
dynamic["_irwin_04a"]=norm_irwin(dynamic["_irwin_norm"])
required_irwins=set(dynamic["_irwin_04a"].dropna())
nirops=load_nirops(NIROPS_SOURCE_RESOLVED, required_irwins)

nirops_ids=set(nirops["_irwin_04a"].dropna())
missing=required_irwins-nirops_ids
print("NIROPS rows after incident/time union:", f"{len(nirops):,}")
print("NIROPS incidents retained:", nirops["_irwin_04a"].nunique())
print("Frozen incidents required:", len(required_irwins))
print("Frozen incidents missing:", len(missing))
if missing:
    display(pd.DataFrame({"missing_irwin":sorted(missing)[:100]}))
    raise RuntimeError("Some frozen incidents are missing after correct IRWIN normalization.")
print("NIROPS matching: PASS")


## 4. Helpers: timestamp decoding and arc-length sample weights

In [ ]:
def decode_times(raw):
    a=np.asarray(raw,dtype=np.int64)
    f=a[a!=np.iinfo(np.int64).min]
    mag=float(np.median(np.abs(f.astype(float))))
    unit="ns" if mag>=1e17 else "us" if mag>=1e14 else "ms" if mag>=1e11 else "s"
    t=pd.DatetimeIndex(pd.to_datetime(a,unit=unit,utc=True))
    if not t.is_monotonic_increasing:
        raise RuntimeError("Trajectory times are not monotonic.")
    return t,unit

def part_lengths(units):
    e=units[["unit_row_index","geometry"]].set_index("unit_row_index").explode(index_parts=True).reset_index()
    if "level_1" in e.columns:
        e=e.rename(columns={"level_1":"part_index"})
    elif "part_index" not in e.columns:
        others=[c for c in e.columns if c not in {"unit_row_index","geometry"}]
        if len(others)!=1: raise RuntimeError("Could not identify part index.")
        e=e.rename(columns={others[0]:"part_index"})
    e["part_index"]=e["part_index"].astype(np.int16)
    e["part_length_m"]=e.geometry.length.astype(float)
    return e[["unit_row_index","part_index","part_length_m"]]

def add_weights(units,samples):
    pl=part_lengths(units)
    n=(samples.groupby(["unit_row_index","part_index"],as_index=False)
       .size().rename(columns={"size":"samples_in_part"}))
    x=(samples.merge(pl,on=["unit_row_index","part_index"],how="left",validate="many_to_one")
       .merge(n,on=["unit_row_index","part_index"],how="left",validate="many_to_one"))
    if x["part_length_m"].isna().any(): raise RuntimeError("Sample-to-part weighting failed.")
    x["sample_weight_m"]=x["part_length_m"]/x["samples_in_part"]
    return x

## 5. Helpers: weighted statistics

In [ ]:
def wmean(v,w):
    v=np.asarray(v,float); w=np.asarray(w,float)
    m=np.isfinite(v)&np.isfinite(w)&(w>0)
    return float(np.average(v[m],weights=w[m])) if m.any() else np.nan

def wq(v,w,q):
    v=np.asarray(v,float); w=np.asarray(w,float)
    m=np.isfinite(v)&np.isfinite(w)&(w>0)
    if not m.any(): return np.nan
    v=v[m]; w=w[m]
    o=np.argsort(v); v=v[o]; w=w[o]
    c=np.cumsum(w); target=q*c[-1]
    i=min(np.searchsorted(c,target,side="left"),len(v)-1)
    return float(v[i])

def wfrac(cond,w):
    cond=np.asarray(cond,bool); w=np.asarray(w,float)
    m=np.isfinite(w)&(w>0)
    return float(w[m&cond].sum()/w[m].sum()) if m.any() else np.nan

def dist_summary(d,w,prefix):
    d=np.asarray(d,float)
    return {
        f"{prefix}_mean_signed_distance_m":wmean(d,w),
        f"{prefix}_median_signed_distance_m":wq(d,w,.5),
        f"{prefix}_p10_signed_distance_m":wq(d,w,.1),
        f"{prefix}_p90_signed_distance_m":wq(d,w,.9),
        f"{prefix}_min_signed_distance_m":float(np.nanmin(d)) if np.isfinite(d).any() else np.nan,
        f"{prefix}_inside_fraction":wfrac(d<0,w),
        f"{prefix}_within_50m_fraction":wfrac((d>=0)&(d<=CHALLENGE_TOLERANCE_M),w),
        f"{prefix}_outside_gt50m_fraction":wfrac(d>CHALLENGE_TOLERANCE_M,w),
    }

def lintrend(x,y):
    x=np.asarray(x,float); y=np.asarray(y,float)
    m=np.isfinite(x)&np.isfinite(y); x=x[m]; y=y[m]
    if len(x)<2: return np.nan,np.nan
    xc=x-x.mean(); den=np.sum(xc**2)
    if den<=0: return np.nan,np.nan
    slope=np.sum(xc*(y-y.mean()))/den
    pred=(y.mean()-slope*x.mean())+slope*x
    ssr=np.sum((y-pred)**2); sst=np.sum((y-y.mean())**2)
    return float(slope), float(1-ssr/sst if sst>0 else 1.0)

def eta(distance,rate):
    if not np.isfinite(distance) or not np.isfinite(rate) or rate<=0: return np.nan
    return float(max(distance,0)/rate)

def put_eta(row,name,value):
    row[name]=value
    row[name+"_cap18"]=min(value,18.0) if np.isfinite(value) else np.nan
    row[name+"_cap48"]=min(value,48.0) if np.isfinite(value) else np.nan

## 6. Helpers: approach direction and fireline orientation

In [ ]:
def coords_array(g):
    pts=[]
    if isinstance(g,LineString):
        pts.extend(g.coords)
    elif isinstance(g,MultiLineString):
        for p in g.geoms: pts.extend(p.coords)
    else:
        try:
            for p in g.geoms:
                if isinstance(p,LineString): pts.extend(p.coords)
        except Exception: pass
    return np.asarray(pts,float)[:,:2] if len(pts)>=2 else np.empty((0,2))

def line_orientation(g):
    xy=coords_array(g)
    if len(xy)<2: return np.nan
    z=xy-xy.mean(axis=0)
    cov=np.cov(z.T)
    if cov.shape!=(2,2) or not np.isfinite(cov).all(): return np.nan
    val,vec=np.linalg.eigh(cov); v=vec[:,np.argmax(val)]
    return math.degrees(math.atan2(float(v[0]),float(v[1])))%180.0

def azimuth(dx,dy):
    if not np.isfinite(dx) or not np.isfinite(dy) or math.hypot(dx,dy)<1e-9: return np.nan
    return math.degrees(math.atan2(dx,dy))%360.0

def angle_to_line(direction,line_deg):
    if not np.isfinite(direction) or not np.isfinite(line_deg): return np.nan
    d=abs((direction%180)-line_deg)
    return float(min(d,180-d))

def shortest_vectors(fire,geoms):
    arr=np.asarray(list(geoms),dtype=object)
    rep=np.empty(len(arr),dtype=object); rep[:]=fire
    if hasattr(shapely,"shortest_line"):
        con=shapely.shortest_line(rep,arr)
        d=np.asarray(shapely.length(con),float)
        a=shapely.get_point(con,0); b=shapely.get_point(con,-1)
        dx=np.asarray(shapely.get_x(b)-shapely.get_x(a),float)
        dy=np.asarray(shapely.get_y(b)-shapely.get_y(a),float)
        return d,dx,dy
    from shapely.ops import nearest_points
    D=[]; DX=[]; DY=[]
    for g in arr:
        p,q=nearest_points(fire,g); x=q.x-p.x; y=q.y-p.y
        D.append(math.hypot(x,y)); DX.append(x); DY.append(y)
    return np.asarray(D),np.asarray(DX),np.asarray(DY)

## 7. Resolve per-incident shards

In [ ]:
def resolve_shard(root,raw_irwin,suffix):
    direct=root/f"{raw_irwin}{suffix}"
    if direct.exists(): return direct
    target=norm_irwin(pd.Series([raw_irwin])).iloc[0]
    matches=[]
    for p in root.glob(f"*{suffix}"):
        if norm_irwin(pd.Series([p.stem])).iloc[0]==target:
            matches.append(p)
    if len(matches)!=1:
        raise RuntimeError(f"Could not uniquely resolve shard for {raw_irwin} under {root}; matches={len(matches)}")
    return matches[0]

def earliest_supported(point_evidence):
    required={"unit_uid","point_evidence","event_start_idx"}
    missing=required-set(point_evidence.columns)
    if missing:
        raise RuntimeError(f"Point-evidence shard missing columns: {sorted(missing)}")
    labels=point_evidence["point_evidence"].astype("string").str.lower()
    idx=pd.to_numeric(point_evidence["event_start_idx"], errors="coerce")
    mask=(labels.str.contains("supported",na=False) &
          ~labels.str.contains("conflict",na=False) &
          (idx>=0))
    x=point_evidence.loc[mask,["unit_uid","event_start_idx"]].copy()
    x["event_start_idx"]=pd.to_numeric(x["event_start_idx"],errors="coerce")
    return x.groupby("unit_uid")["event_start_idx"].min().astype("Int64") if len(x) else pd.Series(dtype="Int64")


## 8. Correct 03L WKB geometry reader

In [ ]:
def read_03l_units(path):
    """Read ordinary 03L Parquet and decode the WKB-byte geometry column."""
    df=pd.read_parquet(path)
    if "geometry" not in df.columns:
        raise RuntimeError(f"No geometry column in 03L unit shard: {path}")
    nonnull=df["geometry"].dropna()
    if len(nonnull)==0:
        raise RuntimeError(f"Empty geometry column in 03L unit shard: {path}")
    example=nonnull.iloc[0]
    if not isinstance(example,(bytes,bytearray,memoryview)):
        raise RuntimeError(f"Expected WKB bytes, found {type(example)} in {path}")
    geometry=gpd.GeoSeries.from_wkb(df["geometry"],crs=CRS_METRIC)
    attrs=df.drop(columns=["geometry"])
    return gpd.GeoDataFrame(attrs,geometry=geometry,crs=CRS_METRIC)

print("03L WKB reader ready.")


## 9. Per-incident feature construction

In [ ]:
def build_incident(fz, prog):

    raw = str(
        fz["_irwin_norm"].iloc[0]
    )

    # ============================================================
    # RESOLVE INPUT SHARDS
    # ============================================================

    up = resolve_shard(
        UNIT_ROOT,
        raw,
        ".parquet",
    )

    sp = resolve_shard(
        SAMPLE_ROOT,
        raw,
        ".parquet",
    )

    ep = resolve_shard(
        POINT_EVIDENCE_ROOT,
        raw,
        ".parquet",
    )

    tp = resolve_shard(
        TRAJECTORY_ROOT,
        raw,
        ".npz",
    )


    # ============================================================
    # LOAD INCIDENT DATA
    # ============================================================

    # 03L units are ordinary Parquet containing WKB bytes.
    units = read_03l_units(up)

    # Sample table does not require GeoPandas.
    samples = pd.read_parquet(sp)

    pe = pd.read_parquet(ep)

    keep = set(
        fz["unit_uid"]
    )

    units = units[
        units["unit_uid"].isin(keep)
    ].copy()

    samples = samples[
        samples["unit_uid"].isin(keep)
    ].copy()

    pe = pe[
        pe["unit_uid"].isin(keep)
    ].copy()


    # ============================================================
    # CRS
    # ============================================================

    if str(units.crs).upper() != CRS_METRIC:

        units = units.to_crs(
            CRS_METRIC
        )


    # ============================================================
    # ARC-LENGTH SAMPLE WEIGHTS
    # ============================================================

    samples = add_weights(
        units,
        samples,
    )


    # ============================================================
    # LOAD CACHED SIGNED-DISTANCE TRAJECTORY
    # ============================================================

    with np.load(
        tp,
        allow_pickle=False,
    ) as z:

        signed = np.asarray(
            z["signed"],
            dtype=np.float32,
        )

        times, ts_unit = decode_times(
            z["times_ns"].copy()
        )


    sample_matrix_rows = (
        samples[
            "sample_row_index"
        ]
        .to_numpy(
            dtype=int
        )
    )

    if len(sample_matrix_rows) == 0:

        raise RuntimeError(
            f"No sample points available for incident {raw}."
        )

    if (
        sample_matrix_rows.min() < 0
        or sample_matrix_rows.max() >= signed.shape[0]
    ):

        raise RuntimeError(
            "sample_row_index outside signed matrix."
        )

    if signed.shape[1] != len(times):

        raise RuntimeError(
            "Trajectory time dimension mismatch."
        )


    # ============================================================
    # VERIFY NIROPS ↔ CACHE TEMPORAL ALIGNMENT
    # ============================================================

    prog = (
        prog.sort_values(
            "_time_04a"
        )
        .reset_index(
            drop=True
        )
    )

    pt = pd.DatetimeIndex(
        prog["_time_04a"]
    )

    if len(pt) != len(times):

        raise RuntimeError(
            f"NIROPS/cache observation mismatch: "
            f"{len(pt)} vs {len(times)}"
        )


    sec = np.abs(
        (
            pd.Series(pt)
            .reset_index(drop=True)
            -
            pd.Series(times)
            .reset_index(drop=True)
        )
        .dt
        .total_seconds()
        .to_numpy()
    )

    if np.nanmax(sec) > 1:

        raise RuntimeError(
            "NIROPS/cache timestamps differ by >1 second."
        )


    # ============================================================
    # RECONSTRUCT PREDICTION ANCHOR
    # ============================================================

    f = fz.copy()

    if "earliest_supported_start_idx" in f.columns:

        f["prediction_anchor_idx"] = (
            pd.to_numeric(
                f[
                    "earliest_supported_start_idx"
                ],
                errors="coerce",
            )
            .astype("Int64")
        )

    else:

        emap = earliest_supported(
            pe
        )

        f["prediction_anchor_idx"] = (
            f["unit_uid"]
            .map(emap)
            .astype("Int64")
        )


    # ============================================================
    # DOMINANT CHALLENGE START
    # ============================================================

    domcol = next(
        (
            c
            for c in [
                "dominant_supported_start_idx",
                "dominant_supported_start_idx_timing",
                "primary_event_start_idx",
                "primary_start_idx",
                "challenge_start_idx",
            ]
            if c in f.columns
        ),
        None,
    )

    if domcol is not None:

        f["dominant_anchor_idx"] = (
            pd.to_numeric(
                f[domcol],
                errors="coerce",
            )
            .astype("Int64")
        )

    else:

        # If 03N has no separate dominant index field,
        # use earliest anchor only for comparison bookkeeping.
        f["dominant_anchor_idx"] = (
            f["prediction_anchor_idx"]
        )


    # ============================================================
    # LOOKUP STRUCTURES
    # ============================================================

    sg = {
        uid: group
        for uid, group
        in samples.groupby(
            "unit_uid",
            sort=False,
        )
    }

    geom = (
        units
        .set_index(
            "unit_uid"
        )[
            "geometry"
        ]
        .to_dict()
    )

    orient = {
        uid: line_orientation(
            geometry
        )
        for uid, geometry
        in geom.items()
    }


    # ============================================================
    # PER-UNIT FEATURES
    # ============================================================

    out = []

    for rec in f.to_dict(
        "records"
    ):

        uid = rec["unit_uid"]

        a = rec.get(
            "prediction_anchor_idx",
            pd.NA,
        )

        d = rec.get(
            "dominant_anchor_idx",
            pd.NA,
        )


        # --------------------------------------------------------
        # No usable prediction anchor
        # --------------------------------------------------------

        if pd.isna(a):

            out.append(
                {
                    "unit_uid": uid,
                    "_irwin_norm": rec[
                        "_irwin_norm"
                    ],
                    "IncidentName": rec.get(
                        "IncidentName"
                    ),
                    "Year": rec.get(
                        "Year"
                    ),
                    "prediction_anchor_available": False,
                }
            )

            continue


        a = int(a)

        d = (
            int(d)
            if not pd.isna(d)
            else a
        )


        if (
            a < 0
            or a >= len(times)
        ):

            raise RuntimeError(
                f"Bad prediction anchor index {a} "
                f"for unit {uid}"
            )


        # Dominant timing is diagnostic only.
        # Fall back safely if its index is malformed.
        if (
            d < 0
            or d >= len(times)
        ):

            d = a


        if uid not in sg:

            raise RuntimeError(
                f"No sample rows found for unit {uid}"
            )


        g = sg[uid]

        r = (
            g[
                "sample_row_index"
            ]
            .to_numpy(
                dtype=int
            )
        )

        w = (
            g[
                "sample_weight_m"
            ]
            .to_numpy(
                dtype=float
            )
        )


        # --------------------------------------------------------
        # FIRE STATE AT PREDICTION ANCHOR
        # --------------------------------------------------------

        x0 = signed[
            r,
            a,
        ].astype(float)


        xd = signed[
            r,
            d,
        ].astype(float)


        row = {

            "unit_uid": uid,

            "_irwin_norm": rec[
                "_irwin_norm"
            ],

            "IncidentName": rec.get(
                "IncidentName"
            ),

            "Year": rec.get(
                "Year"
            ),


            # Prediction timing
            "prediction_anchor_available": True,

            "prediction_anchor_idx": a,

            "prediction_anchor_utc": times[a],


            # Dominant target-alignment timing
            "dominant_anchor_idx": d,

            "dominant_anchor_utc": times[d],

            "prediction_anchor_offset_from_dominant_h": (
                times[d]
                - times[a]
            ).total_seconds()
            / 3600.0,


            # Sampling / unit information
            "sample_points_04a": len(g),

            "unit_actual_line_length_m_04a": float(
                w.sum()
            ),

            "fireline_orientation_deg": orient.get(
                uid,
                np.nan,
            ),


            # Leakage diagnostics
            "dominant_t0_inside_fraction": wfrac(
                xd < 0,
                w,
            ),

            "dominant_t0_any_inside": bool(
                np.any(
                    xd < 0
                )
            ),

            "prediction_t0_any_inside": bool(
                np.any(
                    x0 < 0
                )
            ),

            "prediction_anchor_clean_no_inside": bool(
                not np.any(
                    x0 < 0
                )
            ),

            "trajectory_timestamp_integer_unit": (
                ts_unit
            ),
        }


        # --------------------------------------------------------
        # DISTANCE STATE AT t0
        # --------------------------------------------------------

        row.update(
            dist_summary(
                x0,
                w,
                "t0",
            )
        )


        # ========================================================
        # PREVIOUS NIROPS OBSERVATION
        # ========================================================

        row[
            "has_previous_nirops_observation"
        ] = (
            a >= 1
        )


        if a >= 1:

            xp = signed[
                r,
                a - 1,
            ].astype(float)


            row.update(
                dist_summary(
                    xp,
                    w,
                    "prev",
                )
            )


            dt = (
                times[a]
                - times[a - 1]
            ).total_seconds() / 3600.0


            row[
                "previous_nirops_interval_hours"
            ] = dt


            if dt > 0:

                rates = (
                    xp
                    - x0
                ) / dt


                # Point/sample weighted rates
                row[
                    "approach_rate_last_mean_m_per_h"
                ] = wmean(
                    rates,
                    w,
                )


                row[
                    "approach_rate_last_median_m_per_h"
                ] = wq(
                    rates,
                    w,
                    0.50,
                )


                row[
                    "approach_rate_last_p10_m_per_h"
                ] = wq(
                    rates,
                    w,
                    0.10,
                )


                row[
                    "approach_rate_last_p90_m_per_h"
                ] = wq(
                    rates,
                    w,
                    0.90,
                )


                row[
                    "fraction_line_approaching_last_interval"
                ] = wfrac(
                    rates > 0,
                    w,
                )


                # Unit-summary distance rates
                row[
                    "mean_distance_approach_rate_last_m_per_h"
                ] = (
                    row[
                        "prev_mean_signed_distance_m"
                    ]
                    - row[
                        "t0_mean_signed_distance_m"
                    ]
                ) / dt


                row[
                    "median_distance_approach_rate_last_m_per_h"
                ] = (
                    row[
                        "prev_median_signed_distance_m"
                    ]
                    - row[
                        "t0_median_signed_distance_m"
                    ]
                ) / dt


                row[
                    "p10_distance_approach_rate_last_m_per_h"
                ] = (
                    row[
                        "prev_p10_signed_distance_m"
                    ]
                    - row[
                        "t0_p10_signed_distance_m"
                    ]
                ) / dt


        # ========================================================
        # SHORT HISTORY
        # ========================================================

        h0 = max(
            0,
            a
            - (
                MAX_HISTORY_OBSERVATIONS
                - 1
            ),
        )


        idx = np.arange(
            h0,
            a + 1,
        )


        th = np.array(
            [
                (
                    times[i]
                    - times[a]
                ).total_seconds()
                / 3600.0
                for i in idx
            ],
            dtype=float,
        )


        dm = []
        dmed = []


        for i in idx:

            xx = signed[
                r,
                i,
            ].astype(float)


            dm.append(
                wmean(
                    xx,
                    w,
                )
            )


            dmed.append(
                wq(
                    xx,
                    w,
                    0.50,
                )
            )


        dm = np.asarray(
            dm,
            dtype=float,
        )


        dmed = np.asarray(
            dmed,
            dtype=float,
        )


        sm, r2m = lintrend(
            th,
            dm,
        )


        sd, r2d = lintrend(
            th,
            dmed,
        )


        row[
            "history_observation_count"
        ] = len(idx)


        row[
            "history_span_hours"
        ] = (
            -th[0]
            if len(th)
            else np.nan
        )


        row[
            "history_mean_distance_approach_rate_m_per_h"
        ] = (
            -sm
            if np.isfinite(sm)
            else np.nan
        )


        row[
            "history_mean_distance_trend_r2"
        ] = r2m


        row[
            "history_median_distance_approach_rate_m_per_h"
        ] = (
            -sd
            if np.isfinite(sd)
            else np.nan
        )


        row[
            "history_median_distance_trend_r2"
        ] = r2d


        # ========================================================
        # INTERVAL-TO-INTERVAL HISTORY
        # ========================================================

        irates = []


        for j in range(
            1,
            len(idx),
        ):

            dt_hist = (
                th[j]
                - th[j - 1]
            )


            if dt_hist > 0:

                irates.append(
                    (
                        dm[j - 1]
                        - dm[j]
                    )
                    / dt_hist
                )


        if irates:

            irates = np.asarray(
                irates,
                dtype=float,
            )


            row[
                "history_interval_approach_rate_mean_m_per_h"
            ] = float(
                np.mean(
                    irates
                )
            )


            row[
                "history_interval_approach_rate_std_m_per_h"
            ] = (
                float(
                    np.std(
                        irates
                    )
                )
                if len(irates) > 1
                else 0.0
            )


            row[
                "history_approach_interval_fraction"
            ] = float(
                np.mean(
                    irates > 0
                )
            )


            if len(irates) >= 2:

                row[
                    "approach_rate_change_last_minus_previous_m_per_h"
                ] = float(
                    irates[-1]
                    - irates[-2]
                )


                if len(th) >= 3:

                    dt1 = (
                        th[-1]
                        - th[-2]
                    )

                    dt0 = (
                        th[-2]
                        - th[-3]
                    )


                    midpoint_gap = (
                        0.5
                        * (
                            dt1
                            + dt0
                        )
                    )


                    row[
                        "approx_approach_acceleration_m_per_h2"
                    ] = (
                        (
                            irates[-1]
                            - irates[-2]
                        )
                        / midpoint_gap
                        if midpoint_gap > 0
                        else np.nan
                    )


        # ========================================================
        # ETA CANDIDATES
        # ========================================================

        put_eta(
            row,
            "eta_line_from_last_mean_h",
            eta(
                row[
                    "t0_mean_signed_distance_m"
                ],
                row.get(
                    "mean_distance_approach_rate_last_m_per_h",
                    np.nan,
                ),
            ),
        )


        put_eta(
            row,
            "eta_line_from_last_median_h",
            eta(
                row[
                    "t0_median_signed_distance_m"
                ],
                row.get(
                    "median_distance_approach_rate_last_m_per_h",
                    np.nan,
                ),
            ),
        )


        put_eta(
            row,
            "eta_line_from_last_p10_h",
            eta(
                row[
                    "t0_p10_signed_distance_m"
                ],
                row.get(
                    "p10_distance_approach_rate_last_m_per_h",
                    np.nan,
                ),
            ),
        )


        put_eta(
            row,
            "eta_line_from_history_mean_h",
            eta(
                row[
                    "t0_mean_signed_distance_m"
                ],
                row.get(
                    "history_mean_distance_approach_rate_m_per_h",
                    np.nan,
                ),
            ),
        )


        # ========================================================
        # OBSERVED FIRE GEOMETRY CONTEXT AT t0
        # ========================================================

        fg = (
            prog
            .iloc[a]
            .geometry
        )


        row[
            "observed_fire_area_t0_km2"
        ] = (
            fg.area
            / 1e6
        )


        row[
            "observed_fire_perimeter_t0_km"
        ] = (
            fg.boundary.length
            / 1000.0
        )


        out.append(
            row
        )


    # ============================================================
    # CREATE INCIDENT FEATURE FRAME
    # ============================================================

    feat = pd.DataFrame(
        out
    )


    # ============================================================
    # LOCAL APPROACH DIRECTION
    # ============================================================

    glookup = (
        units
        .set_index(
            "unit_uid"
        )[
            "geometry"
        ]
    )


    direction = []


    if (
        "prediction_anchor_idx"
        in feat.columns
        and "prediction_anchor_available"
        in feat.columns
    ):

        valid_anchor_rows = feat[
            feat[
                "prediction_anchor_available"
            ].fillna(False)
        ]


        for ai, grp in valid_anchor_rows.groupby(
            "prediction_anchor_idx",
            sort=False,
        ):

            ai = int(ai)

            fg = (
                prog
                .iloc[ai]
                .geometry
            )


            uids = (
                grp[
                    "unit_uid"
                ]
                .tolist()
            )


            geoms = [
                glookup.loc[u]
                for u in uids
            ]


            dist, dx, dy = shortest_vectors(
                fg,
                geoms,
            )


            for (
                u,
                dd,
                vx,
                vy,
            ) in zip(
                uids,
                dist,
                dx,
                dy,
            ):

                az = azimuth(
                    vx,
                    vy,
                )


                lo = orient.get(
                    u,
                    np.nan,
                )


                direction.append(
                    {
                        "unit_uid": u,

                        "local_fire_to_line_distance_geom_m": float(
                            dd
                        ),

                        "approach_vector_dx_m": float(
                            vx
                        ),

                        "approach_vector_dy_m": float(
                            vy
                        ),

                        "approach_azimuth_deg": az,

                        "approach_vs_line_angle_deg": angle_to_line(
                            az,
                            lo,
                        ),

                        "approach_direction_reliable": bool(
                            np.isfinite(dd)
                            and dd
                            >= DIRECTION_MIN_DISTANCE_M
                            and np.isfinite(az)
                        ),
                    }
                )


    if direction:

        feat = feat.merge(
            pd.DataFrame(
                direction
            ),
            on="unit_uid",
            how="left",
            validate="one_to_one",
        )


    # ============================================================
    # GUARANTEE OPTIONAL COLUMNS EXIST
    # ============================================================
    #
    # Some prediction anchors occur at the first NIROPS observation.
    # In that case there is no t_-1 and therefore no previous-
    # interval approach-rate information.
    #
    # This is legitimate missing observational history.
    # Represent it as NaN / False rather than failing.
    # ============================================================

    optional_numeric_columns = [

        # Previous NIROPS state
        "prev_mean_signed_distance_m",
        "prev_median_signed_distance_m",
        "prev_p10_signed_distance_m",
        "prev_p90_signed_distance_m",
        "prev_min_signed_distance_m",
        "prev_inside_fraction",
        "prev_within_50m_fraction",
        "prev_outside_gt50m_fraction",
        "previous_nirops_interval_hours",

        # Last-interval point/sample approach rates
        "approach_rate_last_mean_m_per_h",
        "approach_rate_last_median_m_per_h",
        "approach_rate_last_p10_m_per_h",
        "approach_rate_last_p90_m_per_h",
        "fraction_line_approaching_last_interval",

        # Unit-level last-interval approach rates
        "mean_distance_approach_rate_last_m_per_h",
        "median_distance_approach_rate_last_m_per_h",
        "p10_distance_approach_rate_last_m_per_h",

        # History-derived quantities
        "history_interval_approach_rate_mean_m_per_h",
        "history_interval_approach_rate_std_m_per_h",
        "history_approach_interval_fraction",
        "approach_rate_change_last_minus_previous_m_per_h",
        "approx_approach_acceleration_m_per_h2",

        # Direction
        "local_fire_to_line_distance_geom_m",
        "approach_vector_dx_m",
        "approach_vector_dy_m",
        "approach_azimuth_deg",
        "approach_vs_line_angle_deg",
    ]


    for column in optional_numeric_columns:

        if column not in feat.columns:

            feat[column] = np.nan


    optional_boolean_columns = [

        "prediction_anchor_available",

        "prediction_anchor_clean_no_inside",

        "prediction_t0_any_inside",

        "dominant_t0_any_inside",

        "has_previous_nirops_observation",

        "approach_direction_reliable",
    ]


    for column in optional_boolean_columns:

        if column not in feat.columns:

            feat[column] = False


    # ============================================================
    # FEATURE READINESS
    # ============================================================

    feat[
        "approach_rate_available"
    ] = (
        feat[
            "has_previous_nirops_observation"
        ]
        .fillna(False)
        &
        feat[
            "mean_distance_approach_rate_last_m_per_h"
        ]
        .notna()
    )


    feat[
        "operational_anchor_clean"
    ] = (
        feat[
            "prediction_anchor_available"
        ]
        .fillna(False)
        &
        feat[
            "prediction_anchor_clean_no_inside"
        ]
        .fillna(False)
    )


    feat[
        "operational_dynamic_feature_ready"
    ] = (
        feat[
            "operational_anchor_clean"
        ]
        &
        feat[
            "approach_rate_available"
        ]
    )


    # ============================================================
    # INCIDENT METADATA
    # ============================================================

    meta = {

        "_irwin_norm": raw,

        "units": len(
            feat
        ),

        "clean_anchor_units": int(
            feat[
                "operational_anchor_clean"
            ].sum()
        ),

        "units_with_previous_nirops": int(
            feat[
                "has_previous_nirops_observation"
            ].sum()
        ),

        "approach_rate_available_units": int(
            feat[
                "approach_rate_available"
            ].sum()
        ),

        "dynamic_feature_ready_units": int(
            feat[
                "operational_dynamic_feature_ready"
            ].sum()
        ),

        "dominant_anchor_any_inside_units": int(
            feat[
                "dominant_t0_any_inside"
            ]
            .fillna(False)
            .sum()
        ),

        "earliest_anchor_any_inside_units": int(
            feat[
                "prediction_t0_any_inside"
            ]
            .fillna(False)
            .sum()
        ),

        "timestamp_integer_unit": (
            ts_unit
        ),
    }


    return feat, meta

## 10. One-incident preflight

Run this before the 382-incident production cell. It verifies WKB decoding, point/sample alignment, cached trajectory timestamps, NIROPS timestamps, anchor reconstruction, and feature construction on one real incident.

In [ ]:
nirops_groups={k:g for k,g in nirops.groupby("_irwin_04a",sort=False)}

TEST_IRWIN=dynamic["_irwin_norm"].iloc[0]
test_frozen=dynamic[dynamic["_irwin_norm"]==TEST_IRWIN].copy()
test_key=test_frozen["_irwin_04a"].iloc[0]

print("Preflight incident:",TEST_IRWIN)
print("Frozen timing-eligible units:",len(test_frozen))

test_features,test_metadata=build_incident(test_frozen,nirops_groups[test_key])
assert len(test_features)==len(test_frozen)
assert test_features["unit_uid"].is_unique

display(pd.DataFrame([test_metadata]))
display(test_features.head())
print("ONE-INCIDENT PREFLIGHT: PASS")


## 11. Resumable 382-incident V2 production

In [ ]:
RUN_LOG=OUTPUT_ROOT/"04A_V2_RUN_LOG.csv"
nirops_groups={k:g for k,g in nirops.groupby("_irwin_04a",sort=False)}
run=[]

groups=list(dynamic.groupby("_irwin_norm",sort=False))
bar=tqdm(groups,total=len(groups),desc="04A approach incidents")

for raw,fz in bar:
    start=time.perf_counter()
    out=SHARD_ROOT/f"{raw}.parquet"
    try:
        if RESUME and out.exists():
            p=pd.read_parquet(out,columns=["unit_uid","operational_anchor_clean","operational_dynamic_feature_ready"])
            run.append({"_irwin_norm":raw,"status":"SKIPPED_COMPLETE","units":len(p),
                        "clean_anchor_units":int(p["operational_anchor_clean"].sum()),
                        "dynamic_feature_ready_units":int(p["operational_dynamic_feature_ready"].sum()),
                        "runtime_seconds":time.perf_counter()-start})
            bar.set_postfix({"incident":str(raw)[:10],"status":"resume"})
            continue

        k=str(fz["_irwin_04a"].iloc[0])
        feat,meta=build_incident(fz,nirops_groups[k])
        feat.to_parquet(out,index=False)
        meta.update({"status":"OK","runtime_seconds":time.perf_counter()-start})
        run.append(meta)
        bar.set_postfix({"incident":str(raw)[:10],"units":meta["units"],"clean":meta["clean_anchor_units"],"sec":f'{meta["runtime_seconds"]:.2f}'})
        del feat; gc.collect()

    except Exception as e:
        (OUTPUT_ROOT/f"ERROR_{raw}.txt").write_text(traceback.format_exc(),encoding="utf-8")
        run.append({"_irwin_norm":raw,"status":"ERROR","error":f"{type(e).__name__}: {e}",
                    "runtime_seconds":time.perf_counter()-start})
        bar.set_postfix({"incident":str(raw)[:10],"status":"ERROR"})

    pd.DataFrame(run).to_csv(RUN_LOG,index=False)

## 12. Combine V2 incident shards

In [ ]:
files=sorted(SHARD_ROOT.glob("*.parquet"))
expected=dynamic["_irwin_norm"].nunique()
print("Completed shards:",len(files),"/",expected)

if len(files)!=expected:
    log=pd.read_csv(RUN_LOG)
    display(log[log["status"]=="ERROR"])
    raise RuntimeError("04A_v2 incomplete. Fix errors and rerun; completed shards will be reused.")

approach=pd.concat([pd.read_parquet(p) for p in tqdm(files,desc="Combine 04A shards")],ignore_index=True)
assert approach["unit_uid"].is_unique
assert len(approach)==len(dynamic)

FEATURES_PATH=PROCESSED_ROOT/"dynamic_approach_features_v2.parquet"
approach.to_parquet(FEATURES_PATH,index=False)
print("Rows:",f"{len(approach):,}")
print("Saved:",FEATURES_PATH)

## 13. Prediction-anchor audit

In [ ]:
qa=approach.merge(
    dynamic[["unit_uid","_engagement_norm","hold_label"]],
    on="unit_uid",how="left",validate="one_to_one"
)

anchor=(qa.groupby("_engagement_norm").agg(
    units=("unit_uid","size"),
    dominant_anchor_any_inside=("dominant_t0_any_inside","sum"),
    earliest_anchor_any_inside=("prediction_t0_any_inside","sum"),
    clean_earliest_anchor=("operational_anchor_clean","sum"),
    has_previous_observation=("has_previous_nirops_observation","sum"),
    operational_dynamic_feature_ready=("operational_dynamic_feature_ready","sum"),
    median_earliest_to_dominant_offset_h=("prediction_anchor_offset_from_dominant_h","median"),
    p90_earliest_to_dominant_offset_h=("prediction_anchor_offset_from_dominant_h",lambda s:s.quantile(.90)),
).reset_index())

for c in ["dominant_anchor_any_inside","earliest_anchor_any_inside","clean_earliest_anchor",
          "has_previous_observation","operational_dynamic_feature_ready"]:
    anchor[c+"_pct"]=anchor[c]/anchor["units"]*100

display(anchor)
anchor.to_csv(OUTPUT_ROOT/"PREDICTION_ANCHOR_AUDIT.csv",index=False)

## 14. Distance, approach-rate, ETA, and direction summaries

In [ ]:
distance=(qa.groupby("_engagement_norm").agg(
    units=("unit_uid","size"),
    median_t0_mean_distance_m=("t0_mean_signed_distance_m","median"),
    p10_t0_mean_distance_m=("t0_mean_signed_distance_m",lambda s:s.quantile(.10)),
    p90_t0_mean_distance_m=("t0_mean_signed_distance_m",lambda s:s.quantile(.90)),
    median_t0_p10_distance_m=("t0_p10_signed_distance_m","median"),
    median_geom_distance_m=("local_fire_to_line_distance_geom_m","median"),
    median_inside_fraction=("t0_inside_fraction","median"),
    median_within50_fraction=("t0_within_50m_fraction","median"),
).reset_index())
display(distance)
distance.to_csv(OUTPUT_ROOT/"DISTANCE_STATE_SUMMARY.csv",index=False)

rates=(qa.groupby("_engagement_norm").agg(
    units=("unit_uid","size"),
    rate_available=("approach_rate_available","sum"),
    median_last_mean_rate_m_per_h=("mean_distance_approach_rate_last_m_per_h","median"),
    p10_last_mean_rate_m_per_h=("mean_distance_approach_rate_last_m_per_h",lambda s:s.quantile(.10)),
    p90_last_mean_rate_m_per_h=("mean_distance_approach_rate_last_m_per_h",lambda s:s.quantile(.90)),
    median_history_rate_m_per_h=("history_mean_distance_approach_rate_m_per_h","median"),
    median_fraction_line_approaching=("fraction_line_approaching_last_interval","median"),
    median_history_r2=("history_mean_distance_trend_r2","median"),
    median_previous_gap_h=("previous_nirops_interval_hours","median"),
).reset_index())
rates["rate_available_pct"]=rates["rate_available"]/rates["units"]*100
display(rates)
rates.to_csv(OUTPUT_ROOT/"APPROACH_RATE_SUMMARY.csv",index=False)

eta_cols=["eta_line_from_last_mean_h","eta_line_from_last_median_h",
          "eta_line_from_last_p10_h","eta_line_from_history_mean_h"]
rows=[]
for label,g in qa.groupby("_engagement_norm"):
    for c in eta_cols:
        v=g[c].dropna()
        rows.append({"_engagement_norm":label,"eta_estimator":c,"units":len(g),"available":len(v),
                     "available_pct":len(v)/len(g)*100,
                     "median_h":v.median() if len(v) else np.nan,
                     "p10_h":v.quantile(.1) if len(v) else np.nan,
                     "p90_h":v.quantile(.9) if len(v) else np.nan,
                     "pct_le_6h":(v<=6).mean()*100 if len(v) else np.nan,
                     "pct_le_18h":(v<=18).mean()*100 if len(v) else np.nan,
                     "pct_le_48h":(v<=48).mean()*100 if len(v) else np.nan})
eta_summary=pd.DataFrame(rows)
display(eta_summary)
eta_summary.to_csv(OUTPUT_ROOT/"ETA_ESTIMATOR_SUMMARY.csv",index=False)

direction=(qa.groupby("_engagement_norm").agg(
    units=("unit_uid","size"),
    direction_available=("approach_azimuth_deg",lambda s:s.notna().sum()),
    reliable_direction=("approach_direction_reliable","sum"),
    median_fire_to_line_geom_distance_m=("local_fire_to_line_distance_geom_m","median"),
    median_approach_vs_line_angle_deg=("approach_vs_line_angle_deg","median"),
    p10_approach_vs_line_angle_deg=("approach_vs_line_angle_deg",lambda s:s.quantile(.1)),
    p90_approach_vs_line_angle_deg=("approach_vs_line_angle_deg",lambda s:s.quantile(.9)),
).reset_index())
direction["reliable_direction_pct"]=direction["reliable_direction"]/direction["units"]*100
display(direction)
direction.to_csv(OUTPUT_ROOT/"APPROACH_DIRECTION_SUMMARY.csv",index=False)

## 15. Candidate operational dynamic cohort

This does **not** replace `FROZEN_V1`.

It is only the subset for which a clean pre-challenge anchor and at least one prior NIROPS observation are available. We review this attrition before using the subset for HRRR extraction.

In [ ]:
candidate=dynamic[[
    "unit_uid","_irwin_norm","IncidentName","Year","_engagement_norm","hold_label"
]].merge(
    approach,on=["unit_uid","_irwin_norm","IncidentName","Year"],
    how="inner",validate="one_to_one"
)

ready=candidate[candidate["operational_dynamic_feature_ready"]].copy()
CANDIDATE_PATH=PROCESSED_ROOT/"operational_dynamic_modeling_cohort_candidate_v2.parquet"
ready.to_parquet(CANDIDATE_PATH,index=False)

cand=(candidate.groupby("_engagement_norm").agg(
    dynamic_frozen_units=("unit_uid","size"),
    clean_anchor_units=("operational_anchor_clean","sum"),
    operational_dynamic_ready_units=("operational_dynamic_feature_ready","sum"),
    incidents=("_irwin_norm","nunique"),
).reset_index())
cand["operational_dynamic_ready_pct"]=cand["operational_dynamic_ready_units"]/cand["dynamic_frozen_units"]*100
display(cand)
cand.to_csv(OUTPUT_ROOT/"CANDIDATE_DYNAMIC_COHORT_SUMMARY.csv",index=False)

print("Candidate operational rows:",f"{len(ready):,}")
print("Saved:",CANDIDATE_PATH)

## 16. Missingness

In [ ]:
idcols={"unit_uid","_irwin_norm","IncidentName","Year"}
miss=pd.DataFrame({"feature":[c for c in approach.columns if c not in idcols]})
miss["missing_count"]=[approach[c].isna().sum() for c in miss["feature"]]
miss["missing_pct"]=miss["missing_count"]/len(approach)*100
miss=miss.sort_values(["missing_pct","feature"],ascending=[False,True])
display(miss.head(70))
miss.to_csv(OUTPUT_ROOT/"FEATURE_MISSINGNESS.csv",index=False)

## 17. Final 04A_v2 report

In [ ]:
def tomd(df):
    try:
        return df.to_markdown(index=False)
    except Exception:
        return df.to_string(index=False)

report = f'''# 04A_v2 Dynamic Pre-Challenge Fire-Approach Feature Audit

Build: `{NOTEBOOK_BUILD_ID}`

Generated: {datetime.now(timezone.utc).isoformat()}

## Input

03N dynamic-timing eligible units: **{len(dynamic):,}**

Incidents: **{dynamic["_irwin_norm"].nunique():,}**

No external data downloaded. No NIROPS rasterization.

## Prediction-anchor audit

{tomd(anchor)}

## Distance state

{tomd(distance)}

## Observed inter-perimeter approach rates

{tomd(rates)}

## Estimated-time-to-line candidates

{tomd(eta_summary)}

## Local approach direction

{tomd(direction)}

## Candidate operational dynamic cohort

{tomd(cand)}

## Interpretation

The 03N target remains frozen and unchanged.

The candidate operational prediction anchor is the earliest supported-event interval start. Any unit already containing sampled line inside the observed perimeter at that time is flagged rather than silently used in a pre-challenge model.

Approach-rate variables are **observed inter-perimeter approach rates**, not true instantaneous rate of spread.

ETA variables use only information at or before the candidate prediction anchor. Actual `t1-t0` is not a predictor.

The local approach azimuth is the shortest vector from the observed NIROPS fire geometry at the prediction anchor toward the corrected local fireline unit. Direction is flagged unreliable when that vector is shorter than {DIRECTION_MIN_DISTANCE_M:.0f} m.

## Next decision

Review:

1. dominant-vs-earliest anchor leakage;
2. attrition from the clean operational anchor;
3. availability and distribution of the four ETA estimators;
4. approach-direction reliability.

After that, freeze the operational feature anchor and choose the ETA estimator for the adaptive HRRR pilot (`04B0`).
'''

REPORT=OUTPUT_ROOT/"DYNAMIC_APPROACH_FEATURE_REPORT_V2.md"
REPORT.write_text(report,encoding="utf-8")
print(report)
print("\nSaved:",REPORT)

## 18. Output inventory

In [ ]:
print("="*88)
print("04A_v2 OUTPUTS")
print("="*88)
print("Features:", FEATURES_PATH)
print("Candidate operational cohort:", CANDIDATE_PATH)
print("Report:", REPORT)
for p in sorted(OUTPUT_ROOT.glob("*")):
    if p.is_file():
        print(f"{p.name:<60}{p.stat().st_size/(1024**2):>10.3f} MB")
print("\nDONE:",NOTEBOOK_BUILD_ID)

In [ ]:
for label, g in qa.groupby("_engagement_norm"):

    x = g[
        "local_fire_to_line_distance_geom_m"
    ].dropna()

    print()
    print("=" * 70)
    print(label)
    print("=" * 70)

    print("N:", f"{len(x):,}")

    print(
        "median:",
        f"{x.quantile(0.50):.1f} m"
    )

    print(
        "p75:",
        f"{x.quantile(0.75):.1f} m"
    )

    print(
        "p90:",
        f"{x.quantile(0.90):.1f} m"
    )

    print(
        "p95:",
        f"{x.quantile(0.95):.1f} m"
    )

    print(
        "p99:",
        f"{x.quantile(0.99):.1f} m"
    )

    print(
        "> 250 m:",
        f"{(x > 250).mean()*100:.2f}%"
    )

    print(
        "> 500 m:",
        f"{(x > 500).mean()*100:.2f}%"
    )

    print(
        "> 1 km:",
        f"{(x > 1000).mean()*100:.2f}%"
    )

    print(
        "> 2 km:",
        f"{(x > 2000).mean()*100:.2f}%"
    )

    print(
        "> 5 km:",
        f"{(x > 5000).mean()*100:.2f}%"
    )